In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Connected to Snowflake successfully!")

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

In [ ]:
df = session.table(
    "ML_PROJECT.ML_SCHEMA.TELCO_CHURN_CLEAN"
).to_pandas()

df.columns = df.columns.str.upper()

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
df = df.drop(columns=["CUSTOMERID"])
df = df.dropna(subset=["CHURN"])

X = df.drop(columns=["CHURN"])
y = df["CHURN"].astype(int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nChurn distribution:")
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [ ]:
numeric_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Preprocessing setup completed!")
print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

In [ ]:
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(
    y_test, y_pred, zero_division=0
), 4))
print("Recall:", round(recall_score(
    y_test, y_pred, zero_division=0
), 4))
print("F1 Score:", round(f1_score(
    y_test, y_pred, zero_division=0
), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
results = X_test.copy()

results["ACTUAL_CHURN"] = y_test.values
results["PREDICTED_CHURN"] = y_pred
results["CHURN_PROBABILITY"] = y_prob

# Reset the DataFrame index
results = results.reset_index(drop=True)

# Save predictions to Snowflake
save_result = session.write_pandas(
    results,
    table_name="CUSTOMER_CHURN_PREDICTIONS",
    database="ML_PROJECT",
    schema="ML_SCHEMA",
    auto_create_table=True,
    overwrite=True
)

print("Predictions saved to Snowflake successfully!")

In [ ]:
session.table(
    "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
).show(10)

In [ ]:
session.table(
    "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
).show(10)

In [ ]:
tables = session.sql("""
    SHOW TABLES LIKE 'CUSTOMER_CHURN_PREDICTIONS'
    IN SCHEMA ML_PROJECT.ML_SCHEMA
""").collect()

print("Table found:", len(tables) > 0)

In [ ]:
print("TEST 123", flush=True)

In [ ]:
print("Checking Snowflake connection...", flush=True)

print("Session:", session, flush=True)

print(
    "Table count:",
    session.table(
        "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
    ).count(),
    flush=True
)

In [ ]:
print("Checking prediction table...", flush=True)

prediction_df = session.table(
    "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
)

row_count = prediction_df.count()

print("Total prediction rows:", row_count, flush=True)

In [ ]:
from IPython.display import display

display(prediction_df.limit(10).to_pandas())

In [ ]:
prediction_df = session.table(
    "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
)

print("Prediction table loaded successfully!", flush=True)
print("Total rows:", prediction_df.count(), flush=True)

In [ ]:
from IPython.display import display

# Connect to the saved prediction table
prediction_df = session.table(
    "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
)

# Check the number of rows
print("Total prediction rows:", prediction_df.count(), flush=True)

# Display the first 10 rows
display(prediction_df.limit(10).to_pandas())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Load saved predictions
prediction_df = session.table(
    "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
).to_pandas()

# Calculate evaluation metrics
y_actual = prediction_df["ACTUAL_CHURN"]
y_predicted = prediction_df["PREDICTED_CHURN"]

print("Accuracy:", round(accuracy_score(y_actual, y_predicted), 4))
print("Precision:", round(precision_score(y_actual, y_predicted, zero_division=0), 4))
print("Recall:", round(recall_score(y_actual, y_predicted, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_actual, y_predicted, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_predicted))

In [ ]:
# Load the saved predictions
prediction_df = session.table(
    "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
).to_pandas()

print("Prediction data loaded successfully!")
print("Total rows:", len(prediction_df))

# Count predicted churn cases
print("\nPredicted churn counts:")
print(prediction_df["PREDICTED_CHURN"].value_counts())

# Calculate percentages
print("\nPredicted churn percentages:")
print(
    (prediction_df["PREDICTED_CHURN"].value_counts(normalize=True) * 100)
    .round(2)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_actual = prediction_df["ACTUAL_CHURN"]
y_predicted = prediction_df["PREDICTED_CHURN"]

print("Accuracy:", round(
    accuracy_score(y_actual, y_predicted), 4
))

print("Precision:", round(
    precision_score(y_actual, y_predicted, zero_division=0), 4
))

print("Recall:", round(
    recall_score(y_actual, y_predicted, zero_division=0), 4
))

print("F1 Score:", round(
    f1_score(y_actual, y_predicted, zero_division=0), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_predicted))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Load predictions directly from Snowflake
prediction_df = session.table(
    "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
).to_pandas()

# Get actual and predicted values
y_actual = prediction_df["ACTUAL_CHURN"]
y_predicted = prediction_df["PREDICTED_CHURN"]

# Calculate metrics
print("Total test records:", len(prediction_df))
print("Accuracy:", round(accuracy_score(y_actual, y_predicted), 4))
print("Precision:", round(precision_score(y_actual, y_predicted, zero_division=0), 4))
print("Recall:", round(recall_score(y_actual, y_predicted, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_actual, y_predicted, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_predicted))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Load saved predictions directly from Snowflake
prediction_df = session.table(
    "ML_PROJECT.ML_SCHEMA.CUSTOMER_CHURN_PREDICTIONS"
).to_pandas()

# Calculate confusion matrix
cm = confusion_matrix(
    prediction_df["ACTUAL_CHURN"],
    prediction_df["PREDICTED_CHURN"]
)

# Display the confusion matrix
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Customer Churn Prediction - Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Get feature names after preprocessing
feature_names = model.named_steps["preprocessing"].get_feature_names_out()

# Get feature importance from Random Forest
importance = model.named_steps["classifier"].feature_importances_

# Create a DataFrame
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

# Select the 10 most important features
importance_df = importance_df.sort_values(
    by="Importance", ascending=False
).head(10)

# Display the results
print(importance_df)

# Plot the results
plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Importance")
plt.title("Top 10 Factors Influencing Customer Churn")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()